# Food-101 and MELD: reproducible training

Run the cells in order. Start with **repo_sample** for a short live demo. Use **custom_sample** to choose class counts and samples, or **full_data** for all 101 Food-101 classes and all official MELD splits.

The notebook trains fresh models, selects checkpoints on validation data, and evaluates on a separate test set. It saves accuracy/loss curves, modality-ratio curves, confusion matrices, macro-F1, per-class accuracy, calibration error, logs, and checkpoints.

Food-101 text is deliberately generated from labels. This demonstrates a controlled shortcut; it is not a natural image-caption benchmark. MELD uses real text and official audio embeddings, without video.


In [ ]:
from pathlib import Path
import os, subprocess, sys
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path('/content/DL_project')
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'balanced_mm/train.py').exists()), Path.cwd() / 'DL_project')
if not (REPO / 'balanced_mm/train.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/Nishant21092004/DL_project.git', str(REPO)], check=True)
PROJECT = REPO / 'balanced_mm'
os.chdir(PROJECT)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
print('Project:', PROJECT)
print('PyTorch:', torch.__version__, '| GPU available:', torch.cuda.is_available())


## Choose the experiment

For the bundled sample, the sample-size fields are ignored: Food-101 has 200/50/50 examples and MELD has 350/70/70 (train/validation/test). Custom samples use the controls below.

In full-data mode, Food-101's original 750 training images per class are divided into 675 training and 75 validation images; all 250 official test images per class are reserved for test. MELD retains its official 9,989/1,109/2,610 splits.

Changing `SEED` changes the custom sample. `SEEDS` controls independent training initialisations. Start with 5 epochs; large runs need more training and compute. Choose a GPU runtime if available.


In [ ]:
DATA_MODE = 'repo_sample' #@param ['repo_sample', 'custom_sample', 'full_data']
DATASET = 'all' #@param ['all', 'food101', 'meld']
DATA_ROOT = 'data' #@param {type:'string'}
FOOD_CLASSES = 'pizza,sushi,hamburger,french_fries,ice_cream,chocolate_cake,ramen,tacos,fried_rice,caesar_salad' #@param {type:'string'}
FOOD_TRAIN_PER_CLASS = 20 #@param {type:'integer'}
FOOD_VAL_PER_CLASS = 5 #@param {type:'integer'}
FOOD_TEST_PER_CLASS = 5 #@param {type:'integer'}
MELD_TRAIN_PER_CLASS = 50 #@param {type:'integer'}
MELD_VAL_PER_CLASS = 10 #@param {type:'integer'}
MELD_TEST_PER_CLASS = 10 #@param {type:'integer'}
FOOD_IMAGE_MAX_SIDE = 96 #@param {type:'integer'}
TEXT_PROMPT_PROBABILITY = 0.7 #@param {type:'number'}
SAMPLE_SEED = 0 #@param {type:'integer'}
EPOCHS = 5 #@param {type:'integer'}
METHODS = 'none,opm,ogm,both' #@param {type:'string'}
SEEDS = '0' #@param {type:'string'}
IMAGE_ENCODER = 'smallcnn' #@param ['smallcnn', 'resnet18']
PRETRAINED_IMAGE = False #@param {type:'boolean'}
TEXT_DIM = 64 #@param {type:'integer'}
TEXT_LAYERS = 1 #@param {type:'integer'}
BATCH_SIZE = 16 #@param {type:'integer'}
LEARNING_RATE = 0.01 #@param {type:'number'}
OUTPUT_DIR = 'demo_runs_new' #@param {type:'string'}

# In custom mode, a per-class count of 0 means all available examples.
# FOOD_CLASSES='all' selects all 101 classes. Image max side 0 keeps original resolution.


## Download or unpack, then prepare the data

`repo_sample` uses real-data ZIP files stored in this repository and needs no large download. The other modes download the official Food-101 archive (~5 GB) and/or MELD feature archive (~923 MB). Re-running reuses complete cached data. Training reads local files after preparation.

To keep raw data across Colab sessions, mount your Google Drive and set `DATA_ROOT` to a folder there before running this cell.


In [ ]:
import json, shlex
root = Path(DATA_ROOT)
DATA_DIR = root / ('demo' if DATA_MODE == 'repo_sample' else 'experiment')
def run_script(name, *arguments):
    command = [sys.executable, 'scripts/' + name, *map(str, arguments)]
    print(shlex.join(command), flush=True)
    subprocess.run(command, check=True)

if DATA_MODE == 'repo_sample':
    run_script('unpack_demo_data.py', '--dataset', DATASET, '--out', DATA_DIR)
else:
    raw = root / 'raw'
    run_script('download_datasets.py', '--dataset', DATASET, '--out', raw)
    full = DATA_MODE == 'full_data'
    run_script('prepare_experiment_data.py', '--dataset', DATASET, '--source', raw, '--out', DATA_DIR,
               '--food-classes', 'all' if full else FOOD_CLASSES,
               '--food-train-per-class', 0 if full else FOOD_TRAIN_PER_CLASS,
               '--food-val-per-class', 75 if full else FOOD_VAL_PER_CLASS,
               '--food-test-per-class', 0 if full else FOOD_TEST_PER_CLASS,
               '--meld-train-per-class', 0 if full else MELD_TRAIN_PER_CLASS,
               '--meld-val-per-class', 0 if full else MELD_VAL_PER_CLASS,
               '--meld-test-per-class', 0 if full else MELD_TEST_PER_CLASS,
               '--food-resize', FOOD_IMAGE_MAX_SIDE, '--text-p', TEXT_PROMPT_PROBABILITY, '--seed', SAMPLE_SEED)


In [ ]:
import csv
from IPython.display import display, Markdown
for dataset in ['food101', 'meld']:
    if DATASET not in ('all', dataset):
        continue
    folder = DATA_DIR / dataset
    manifest = json.loads((folder / 'manifest.json').read_text())
    print(dataset, {split: info['count'] for split, info in manifest['splits'].items()})
    print('Source:', manifest['source'])
    splits = {split: list(csv.DictReader((folder / f'{split}.csv').open())) for split in ['train', 'val', 'test']}
    identifiers = {split: {(row['source_split'], row['source_id']) for row in rows} for split, rows in splits.items()}
    assert not identifiers['train'] & identifiers['val']
    assert not identifiers['train'] & identifiers['test']
    assert not identifiers['val'] & identifiers['test']
    print('Train/validation/test examples are disjoint.')
    for row in splits['train'][:3]:
        print(row)


## Train and evaluate

Each method uses the same data split and each requested seed. Validation chooses the best checkpoint; the test set is evaluated after training. The final partial training batch is retained, so every selected training example is used in each epoch.


In [ ]:
arguments = ['--dataset', DATASET, '--data', DATA_DIR, '--out', OUTPUT_DIR, '--epochs', EPOCHS,
             '--methods', METHODS, '--seeds', SEEDS, '--image-encoder', IMAGE_ENCODER,
             '--text-dim', TEXT_DIM, '--text-layers', TEXT_LAYERS, '--batch-size', BATCH_SIZE, '--lr', LEARNING_RATE]
if PRETRAINED_IMAGE:
    arguments.append('--pretrained-image')
run_script('run_colab_experiments.py', *arguments)


## Results and graphs

Accuracy alone is not enough: compare test macro-F1, class-wise errors, and modality ratios. A smaller modality gap does not guarantee higher accuracy. These short sample runs are demonstrations, not replacements for the older benchmark or final paper experiments.


In [ ]:
from IPython.display import Image as DisplayImage
output = Path(OUTPUT_DIR)
comparison = json.loads((output / 'comparison.json').read_text())
display(Markdown((output / 'RESULTS.md').read_text().split('## Saved evidence')[0]))
display(DisplayImage(filename=str(output / 'comparison.png')))
for record in comparison['runs']:
    folder = output / record['run']
    display(Markdown('### ' + record['run']))
    display(DisplayImage(filename=str(folder / 'training_curves.png')))
    display(DisplayImage(filename=str(folder / 'confusion_matrix.png')))
    metrics = json.loads((folder / 'eval_test.csv.json').read_text())
    print('Per-class accuracy:', metrics['per_class_accuracy'])
    print('Calibration error:', metrics['ece'])


## Download your new results

Includes the metrics, figures, CSV confusion matrices, logs, configuration, and best/last checkpoints. Raw datasets are not included. Save the archive before ending the Colab session.


In [ ]:
import shutil
archive = shutil.make_archive(str(PROJECT.parent / 'experiment_outputs'), 'zip', OUTPUT_DIR)
print('Result archive:', archive)
if IN_COLAB:
    files.download(archive)
